# ⚽ CanchaVision en GPU (Google Colab)

Corre el motor de análisis en una **GPU gratuita** de Colab. Lo que en un portátil sin GPU tarda ~28 min, aquí debería tardar **~1–2 min**.

**Antes de empezar:** menú `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **GPU (T4)**.

Luego ejecuta las celdas en orden (▶️).

## 1. Comprobar que hay GPU

In [ ]:
!nvidia-smi

## 2. Clonar el repo
Edita `REPO_URL` con la URL de tu repositorio de GitHub.

In [ ]:
REPO_URL = "https://github.com/bazoaltoSamuel/canchavision.git"
!git clone $REPO_URL
%cd canchavision

## 3. Instalar dependencias (versión GPU)
Usa `inference-gpu` (onnxruntime en GPU). Tarda un par de minutos.
La última línea fija `supervision` a una versión estable para la homografía.

In [ ]:
!pip install -q -e .
!pip install -q inference-gpu "git+https://github.com/roboflow/sports.git" gdown yt-dlp
!pip install -q "supervision==0.29.1"  # estable para la detección de puntos del campo

## 4. Tu API key de Roboflow
Se pide de forma oculta (no queda guardada en el notebook).

In [ ]:
import os, getpass
os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Roboflow API key: ")

## 5. Elegir el vídeo

**Opción A – link de YouTube.** Pega la URL del partido y elige el tramo.
⚠️ *YouTube suele bloquear las descargas desde Colab ("confirm you're not a bot").
Si falla, usa la Opción B, o pasa cookies (`--cookies cookies.txt`).*

💡 *Funciona mejor con **cámara táctica/elevada**.*

In [ ]:
YOUTUBE_URL = "https://www.youtube.com/watch?v=XXXXXXXXXXX"  # <-- el partido
SECTION = "*10:00-12:00"  # tramo a analizar (mm:ss-mm:ss), ~2 min

STEM = "partido"
VIDEO = f"data/raw/{STEM}.mp4"
!mkdir -p data/raw
!yt-dlp -f "best[height<=720][ext=mp4]/best[height<=720]" --download-sections "$SECTION" --force-keyframes-at-cuts -o "$VIDEO" "$YOUTUBE_URL"
assert os.path.exists(VIDEO), "No se descargó el vídeo (¿YouTube bloqueó? usa la Opción B)"
print("Descargado:", VIDEO)

**Opción B (alternativa)** – clip de muestra de la Bundesliga (Google Drive, sin bloqueo). Ejecuta esta celda en lugar de la de arriba:

In [ ]:
STEM = "2e57b9_0"
VIDEO = f"data/raw/{STEM}.mp4"
!mkdir -p data/raw
!gdown -O "$VIDEO" "https://drive.google.com/uc?id=19PGw55V8aA6GZu5-Aac5_9mCy3fNxmEf"

## 6. Procesar en GPU (y medir el tiempo)
Procesa los primeros ~12 s. Sube `--max-frames` (o bórralo) para más.

In [ ]:
import time
t = time.time()
!canchavision --video "$VIDEO" --config config/football_roboflow.yaml --max-frames 300
print(f"\n⏱️ Tiempo total en GPU: {time.time() - t:.0f} s")

## 7. Ver resultados: pizarrón cenital y mapa de calor

In [ ]:
import cv2
from IPython.display import Image, display
cap = cv2.VideoCapture(f'outputs/{STEM}_radar.mp4')
cap.set(cv2.CAP_PROP_POS_FRAMES, 150)
ok, f = cap.read()
if ok:
    cv2.imwrite('outputs/_radar_frame.png', f)
    print('Pizarrón cenital:'); display(Image('outputs/_radar_frame.png'))
print('Mapa de calor:'); display(Image(f'outputs/{STEM}_heatmap.png'))

## 8. Stats por jugador, posesión y pases

In [ ]:
pos = f"outputs/{STEM}_positions.csv"
ball = f"outputs/{STEM}_ball.csv"
!python scripts/compute_stats.py $pos 25
print('\n' + '='*40)
!python scripts/compute_possession.py $pos $ball

## 9. (Opcional) Guardar resultados en tu Google Drive
Para no perderlos si Colab se desconecta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r outputs "/content/drive/MyDrive/canchavision_outputs"
print('✅ Guardado en tu Drive: canchavision_outputs')